# Pixeloza - demo

Osobny notebook, tylko do demonstracji - modele/generowanie bez zmian, poprawiony frontend (motyw pixel-art, własny układ, dekoracje) i uruchamia Gradio z 3 dropdownami.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q gradio x-transformers

project_path = "/content/drive/MyDrive/Project"


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.4/143.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.6 MB/s eta 0:00:00


### KONFIGURACJA

In [ ]:
WEIGHTS = {
    "vae_model": f"{project_path}/code/vae/model_VQ_VAE.pth",
    "vae_transformer": f"{project_path}/code/vae/vae_encoder.pt",

    "gan_original_model": f"{project_path}/code/gan_original/vqgan.pth",
    "gan_original_transformer": f"{project_path}/code/gan_original/vqgan_encoder.pt",

    "gan_new_model": f"{project_path}/code/gan_new/vqgan_best.pth",
    "gan_new_transformer": f"{project_path}/code/gan_new/vqgan_encoder.pt",

    "diffusion_original": f"{project_path}/code/diffusion_original/diffusion_model_original.pt",
    "diffusion_new": f"{project_path}/code/diffusion_new/diffusion_model_final.pth",
}

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import gradio as gr
import numpy as np
import math
import base64, io
from PIL import Image
from x_transformers import TransformerWrapper, Encoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Używamy: {device}")


Używamy: cpu


## Model VQ-VAE

In [ ]:
class VectorQuantizerVAE(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, commitment_cost):
        super().__init__()
        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1 / num_embeddings, 1 / num_embeddings)
        self.commitment_cost = commitment_cost

    def forward(self, z):
        z = z.permute(0, 2, 3, 1).contiguous()
        z_flat = z.view(-1, z.shape[-1])
        d = (z_flat**2).sum(1, keepdim=True) + (self.embeddings.weight**2).sum(1) \
            - 2 * z_flat @ self.embeddings.weight.t()
        idx = d.argmin(1)
        z_q = self.embeddings(idx).view(z.shape)
        loss = F.mse_loss(z_q.detach(), z) + self.commitment_cost * F.mse_loss(z_q, z.detach())
        z_q = z + (z_q - z).detach()
        return z_q.permute(0, 3, 1, 2).contiguous(), loss, idx


class VQVAE(nn.Module):
    def __init__(self, in_channels=4, hidden_dim=128, num_embeddings=64, embedding_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim // 2, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(hidden_dim // 2, hidden_dim, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(hidden_dim, embedding_dim, 3, 1, 1),
        )
        self.vq_vae = VectorQuantizerVAE(num_embeddings, embedding_dim, commitment_cost=0.25)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embedding_dim, hidden_dim, 3, 1, 1), nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim, hidden_dim // 2, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim // 2, in_channels, 4, 2, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        z_q, vq_loss, idx = self.vq_vae(z)
        return self.decoder(z_q), vq_loss, idx

## Model VQ-GAN

In [ ]:
class VectorQuantizerGANOriginal(nn.Module):
    def __init__(self, num_embeddings, embedding_dim=32, commitment_cost=0.5):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost
        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1 / num_embeddings, 1 / num_embeddings)

    def forward(self, z_e):
        B, C, H, W = z_e.shape
        z_flat = z_e.permute(0, 2, 3, 1).reshape(-1, C)
        d = z_flat.pow(2).sum(1, keepdim=True) - 2 * z_flat @ self.embeddings.weight.t() \
            + self.embeddings.weight.pow(2).sum(1)
        idx = d.argmin(1)
        z_q = self.embeddings(idx).view(B, H, W, C).permute(0, 3, 1, 2)
        vq_loss = F.mse_loss(z_q, z_e.detach()) + self.commitment_cost * F.mse_loss(z_e, z_q.detach())
        z_q = z_e + (z_q - z_e).detach()
        return z_q, vq_loss, idx.view(B, H, W)


class VQGANOriginal(nn.Module):
    def __init__(self, in_channels=4, hidden_dim=64, latent_dim=32, num_embeddings=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(hidden_dim, hidden_dim, 4, 2, 1), nn.ReLU(),
            nn.Conv2d(hidden_dim, latent_dim, 3, 1, 1),
        )
        self.vq_vae = VectorQuantizerGANOriginal(num_embeddings, latent_dim)
        self.decoder = nn.Sequential(
            nn.Conv2d(latent_dim, hidden_dim, 3, 1, 1), nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim, hidden_dim, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim, in_channels, 4, 2, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, vq_loss, idx = self.vq_vae(z_e)
        return self.decoder(z_q), vq_loss, idx

## Model GAN new (ResBlock + spectral norm discriminator)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.GroupNorm(8, channels), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, 3, 1, 1),
            nn.GroupNorm(8, channels), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, 3, 1, 1),
        )

    def forward(self, x):
        return x + self.block(x)


class VectorQuantizerGANNew(nn.Module):
    def __init__(self, num_embeddings=256, embedding_dim=32, commitment_cost=0.25):
        super().__init__()
        self.commitment_cost = commitment_cost
        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1 / num_embeddings, 1 / num_embeddings)

    def forward(self, z_e):
        B, C, H, W = z_e.shape
        z = z_e.permute(0, 2, 3, 1).contiguous().view(-1, C)
        d = (z**2).sum(1, keepdim=True) + (self.embeddings.weight**2).sum(1) \
            - 2 * z @ self.embeddings.weight.t()
        idx = d.argmin(1)
        z_q = self.embeddings(idx).view(B, H, W, C).permute(0, 3, 1, 2).contiguous()
        vq_loss = F.mse_loss(z_q, z_e.detach()) + self.commitment_cost * F.mse_loss(z_e, z_q.detach())
        z_q = z_e + (z_q - z_e).detach()
        return z_q, vq_loss, idx.view(B, H, W)


class VQGANNew(nn.Module):
    def __init__(self):
        super().__init__()
        channels, in_channels, out_channels = 64, 4, 32
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, channels, 4, 2, 1), ResBlock(channels),
            nn.Conv2d(channels, channels, 4, 2, 1), ResBlock(channels),
            nn.Conv2d(channels, out_channels, 3, 1, 1),
        )
        self.vq_vae = VectorQuantizerGANNew(num_embeddings=256, embedding_dim=32)
        self.decoder = nn.Sequential(
            nn.Conv2d(out_channels, channels, 3, 1, 1), ResBlock(channels),
            nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(channels, channels, 3, 1, 1), ResBlock(channels),
            nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(channels, channels, 3, 1, 1),
            nn.GroupNorm(8, channels), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, in_channels, 3, 1, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        z_q, vq_loss, idx = self.vq_vae(z)
        return self.decoder(z_q), vq_loss, idx

## Model Diffusion original i new

In [ ]:
class LinearNoiseScheduler:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cpu"):
        self.timesteps = timesteps
        self.beta = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alpha = 1.0 - self.beta
        self.alpha_cumprod = torch.cumprod(self.alpha, dim=0)
        self.alpha_cumprod_prev = torch.cat([torch.ones(1, device=device), self.alpha_cumprod[:-1]])
        self.posterior_variance = self.beta * (1.0 - self.alpha_cumprod_prev) / (1.0 - self.alpha_cumprod)

    def to(self, device):
        for n in ["beta", "alpha", "alpha_cumprod", "alpha_cumprod_prev", "posterior_variance"]:
            setattr(self, n, getattr(self, n).to(device))
        return self


class SimpleBlockOriginal(nn.Module):
    def __init__(self, in_c, out_c, time_emb_dim):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_c)
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.bn2 = nn.BatchNorm2d(out_c)

    def forward(self, x, t_emb):
        h = F.relu(self.bn1(self.conv1(x)))
        t_proj = F.relu(self.time_mlp(t_emb))[:, :, None, None]
        h = h + t_proj
        return F.relu(self.bn2(self.conv2(h)))


class UNet32Original(nn.Module):
    """inna kolejność argumentów niż w wersji new: n_body, n_glasses, n_head"""
    def __init__(self, n_body, n_glasses, n_head, timesteps, in_channels=4, time_emb_dim=128, base=64):
        super().__init__()
        c = base
        self.time_emb = nn.Embedding(timesteps, time_emb_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_emb_dim, time_emb_dim), nn.GELU(), nn.Linear(time_emb_dim, time_emb_dim))
        self.body_emb = nn.Embedding(n_body, time_emb_dim)
        self.glasses_emb = nn.Embedding(n_glasses, time_emb_dim)
        self.head_emb = nn.Embedding(n_head, time_emb_dim)
        self.cond_mlp = nn.Sequential(nn.Linear(time_emb_dim, time_emb_dim), nn.GELU(), nn.Linear(time_emb_dim, time_emb_dim))
        self.b1 = SimpleBlockOriginal(in_channels, c, time_emb_dim)
        self.down1 = nn.MaxPool2d(2)
        self.b2 = SimpleBlockOriginal(c, 2 * c, time_emb_dim)
        self.down2 = nn.MaxPool2d(2)
        self.mid = SimpleBlockOriginal(2 * c, 2 * c, time_emb_dim)
        self.up2 = nn.ConvTranspose2d(2 * c, c, 2, stride=2)
        self.b3 = SimpleBlockOriginal(3 * c, c, time_emb_dim)
        self.up1 = nn.ConvTranspose2d(c, c // 2, 2, stride=2)
        self.b4 = SimpleBlockOriginal(c // 2 + c, c // 2, time_emb_dim)
        self.final_conv = nn.Conv2d(c // 2, in_channels, 1)
        self.in_channels = in_channels

    def forward(self, x, t, cond):
        c_emb = self.glasses_emb(cond[:, 0]) + self.head_emb(cond[:, 1]) + self.body_emb(cond[:, 2])
        t_emb = self.time_mlp(self.time_emb(t)) + self.cond_mlp(c_emb)
        x1 = self.b1(x, t_emb)
        x2 = self.b2(self.down1(x1), t_emb)
        mid = self.mid(self.down2(x2), t_emb)
        x3 = self.b3(torch.cat([self.up2(mid), x2], 1), t_emb)
        x4 = self.b4(torch.cat([self.up1(x3), x1], 1), t_emb)
        return self.final_conv(x4)


class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=time.device) * -emb)
        emb = time[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)


class SimpleBlockNew(nn.Module):
    def __init__(self, in_c, out_c, time_emb_dim):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_c)
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1)
        self.gn1 = nn.GroupNorm(8, out_c)
        self.gn2 = nn.GroupNorm(8, out_c)

    def forward(self, x, t_emb):
        h = F.silu(self.gn1(self.conv1(x)))
        t_proj = F.silu(self.time_mlp(t_emb))[:, :, None, None]
        h = h + t_proj
        return F.silu(self.gn2(self.conv2(h)))


class UNet32New(nn.Module):
    """inna kolejność argumentów niż w pliku: n_glasses, n_head, n_body"""
    def __init__(self, n_glasses, n_head, n_body, in_channels=4, time_emb_dim=128, base=64):
        super().__init__()
        c = base
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim), nn.GELU(), nn.Linear(time_emb_dim, time_emb_dim),
        )
        self.glasses_emb = nn.Embedding(n_glasses, time_emb_dim)
        self.head_emb = nn.Embedding(n_head, time_emb_dim)
        self.body_emb = nn.Embedding(n_body, time_emb_dim)
        self.cond_mlp = nn.Sequential(nn.Linear(time_emb_dim * 3, time_emb_dim), nn.GELU(), nn.Linear(time_emb_dim, time_emb_dim))
        self.b1 = SimpleBlockNew(in_channels, c, time_emb_dim)
        self.down1 = nn.Conv2d(c, c, 3, stride=2, padding=1)
        self.b2 = SimpleBlockNew(c, 2 * c, time_emb_dim)
        self.down2 = nn.Conv2d(2 * c, 2 * c, 3, stride=2, padding=1)
        self.mid = SimpleBlockNew(2 * c, 2 * c, time_emb_dim)
        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(2 * c, c, 3, padding=1))
        self.b3 = SimpleBlockNew(c + 2 * c, c, time_emb_dim)
        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(c, c // 2, 3, padding=1))
        self.b4 = SimpleBlockNew(c // 2 + c, c // 2, time_emb_dim)
        self.final_conv = nn.Conv2d(c // 2, in_channels, 1)
        self.in_channels = in_channels

    def forward(self, x, t, cond, drop_cond_mask=None):
        c_emb = torch.cat([self.glasses_emb(cond[:, 0]), self.head_emb(cond[:, 1]), self.body_emb(cond[:, 2])], dim=-1)
        if drop_cond_mask is not None:
            c_emb = c_emb * (~drop_cond_mask).unsqueeze(-1)
        t_emb = self.time_mlp(t) + self.cond_mlp(c_emb)
        x1 = self.b1(x, t_emb)
        x2 = self.b2(self.down1(x1), t_emb)
        mid = self.mid(self.down2(x2), t_emb)
        x3 = self.b3(torch.cat([self.up2(mid), x2], 1), t_emb)
        x4 = self.b4(torch.cat([self.up1(x3), x1], 1), t_emb)
        return self.final_conv(x4)

## Wczytanie kategorii (offsety, mapowanie nazw)

In [ ]:
import json, os

tensor_data = torch.load(f"{project_path}/data/processed_images_features.pt", map_location="cpu")
features = tensor_data["features"]

num_glasses = features[:, 0].max().item() + 1
num_head = features[:, 1].max().item() + 1
num_body = features[:, 2].max().item() + 1

offset_glasses = 0
offset_head = offset_glasses + num_glasses
offset_body = offset_head + num_head
offset_image = offset_body + num_body

mapping_path = f"{project_path}/data/label_mappings.json"
if os.path.exists(mapping_path):
    with open(mapping_path) as f:
        real_mapping = json.load(f)
    id_to_name = {
        "glasses": [real_mapping["glasses"][str(i)] for i in range(num_glasses)],
        "head": [real_mapping["head"][str(i)] for i in range(num_head)],
        "body": [real_mapping["body"][str(i)] for i in range(num_body)],
    }
else:
    id_to_name = {
        "glasses": [f"glasses_{i}" for i in range(num_glasses)],
        "head": [f"head_{i}" for i in range(num_head)],
        "body": [f"body_{i}" for i in range(num_body)],
    }
print(f"Kategorie: glasses={num_glasses}, head={num_head}, body={num_body}")

## Rejestr modeli - leniwe wczytywanie (dopiero przy pierwszym użyciu)

In [ ]:
_cache = {}

def _load_vq_masked_transformer(path, seq_len):
    checkpoint = torch.load(path, map_location=device)
    cfg = checkpoint["model_config"]
    offsets = checkpoint["offsets"]
    encoder_attn = Encoder(dim=cfg["dim"], depth=cfg["depth"], heads=cfg["heads"], attn_flash=True, cross_attend=True)
    net = TransformerWrapper(num_tokens=offsets["total_vocab_size"], max_seq_len=seq_len, attn_layers=encoder_attn).to(device)
    net.load_state_dict(checkpoint["transformer_state_dict"])
    net.eval()
    cond_emb = nn.Embedding(offsets["total_vocab_size"], cfg["dim"]).to(device)
    cond_emb.load_state_dict(checkpoint["condition_embedding_state_dict"])
    cond_emb.eval()
    return net, cond_emb, offsets["mask_token_id"]


@torch.no_grad()
def _generate_vq(vq_model, transformer_net, cond_emb,
    mask_token_id,
    seq_len,
    grid_size,
    glasses_id,
    head_id,
    body_id,
    iterations=10,
    temperature=0.7
):
    conditions = torch.tensor([[glasses_id + offset_glasses, head_id + offset_head, body_id + offset_body]],
        dtype=torch.long,
        device=device
    )

    img_tokens = torch.full((1, seq_len), mask_token_id, dtype=torch.long, device=device)
    condition_context = cond_emb(conditions)

    for _ in range(iterations):
        logits = transformer_net(img_tokens,context=condition_context) / temperature
        # Blokujemy tokeny warunkujące
        logits[:, :, :offset_image] = -float("inf")
        # Blokujemy MASK
        logits[:, :, mask_token_id] = -float("inf")
        img_tokens = torch.argmax(logits, dim=-1)

    # 64 tokeny transformera -> 64 tokeny VQGAN
    img_tokens = img_tokens[0] - offset_image

    # Kontrola zakresu tokenów VQGAN.
    if img_tokens.min().item() < 0:
        raise RuntimeError(f"Nieprawidłowy token VQGAN: {img_tokens.min().item()}")

    if img_tokens.max().item() >= vq_model.vq_vae.embeddings.num_embeddings:
        raise RuntimeError(f"Token poza codebookiem VQGAN: {img_tokens.max().item()}")

    # 64 -> 8x8
    img_tokens = img_tokens.view(1, grid_size, grid_size)

    # Tokeny -> embeddingi codebooka VQGAN
    z_q = vq_model.vq_vae.embeddings(img_tokens)

    # [1, 8, 8, 32] -> [1, 32, 8, 8]
    z_q = z_q.permute(0, 3, 1, 2).contiguous()

    # Dekoder VQGAN
    x_recon = vq_model.decoder(z_q)

    return x_recon[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()


def _to_pil(img_np):
    img_uint8 = (img_np[..., :4] * 255).astype(np.uint8)
    return Image.fromarray(img_uint8, mode="RGBA").resize((256, 256), Image.NEAREST)


# _______________________________________________________________
# VAE
def load_vae():
    if "vae" not in _cache:
        m = VQVAE(num_embeddings=64).to(device)
        m.load_state_dict(torch.load(WEIGHTS["vae_model"], map_location=device))
        m.eval()
        net, cond_emb, mask_id = _load_vq_masked_transformer(WEIGHTS["vae_transformer"],seq_len=64)
        _cache["vae"] = (m, net, cond_emb, mask_id)
    return _cache["vae"]


def generate_vae(g, h, b):
    m, net, cond_emb, mask_id = load_vae()
    return _to_pil(_generate_vq(m, net, cond_emb, mask_id, 64, 8, g, h, b))


# _______________________________________________________________
# GAN original
def load_gan_original():
    if "gan_original" not in _cache:
        m = VQGANOriginal(num_embeddings=256).to(device)
        m.load_state_dict(torch.load(WEIGHTS["gan_original_model"], map_location=device))
        m.eval()

        # 256 = rozmiar codebooka
        # 64  = liczba tokenów obrazu (8x8)
        net, cond_emb, mask_id = _load_vq_masked_transformer(WEIGHTS["gan_original_transformer"], seq_len=64)
        _cache["gan_original"] = (m, net, cond_emb, mask_id)

    return _cache["gan_original"]


def generate_gan_original(g, h, b):
    m, net, cond_emb, mask_id = load_gan_original()
    return _to_pil(_generate_vq(m, net, cond_emb, mask_id, 64, 8, g, h, b))


# _______________________________________________________________
# GAN new
def load_gan_new():
    if "gan_new" not in _cache:
        m = VQGANNew().to(device)
        m.load_state_dict(torch.load(WEIGHTS["gan_new_model"], map_location=device))
        m.eval()

        # 256 = rozmiar codebooka
        # 64  = liczba tokenów obrazu (8x8)
        net, cond_emb, mask_id = _load_vq_masked_transformer(WEIGHTS["gan_new_transformer"], seq_len=64)
        _cache["gan_new"] = (m, net, cond_emb, mask_id)

    return _cache["gan_new"]


def generate_gan_new(g, h, b):
    m, net, cond_emb, mask_id = load_gan_new()
    return _to_pil(_generate_vq(m, net, cond_emb, mask_id, 64, 8, g, h, b))


# _______________________________________________________________
# Diffusion original
def load_diffusion_original():
    if "diff_orig" not in _cache:
        sched = LinearNoiseScheduler(timesteps=1000, device=device).to(device)
        m = UNet32Original(num_body, num_glasses, num_head, timesteps=1000).to(device)
        m.load_state_dict(torch.load(WEIGHTS["diffusion_original"], map_location=device))
        m.eval()

        _cache["diff_orig"] = (m, sched)

    return _cache["diff_orig"]


@torch.no_grad()
def generate_diffusion_original(g, h, b):
    m, sched = load_diffusion_original()
    cond = torch.tensor([[g, h, b]], dtype=torch.long, device=device)
    x = torch.randn(1, m.in_channels, 32, 32, device=device)
    for i in reversed(range(sched.timesteps)):
        t = torch.full((1,), i, device=device, dtype=torch.long)
        pred_noise = m(x, t, cond)
        alpha = sched.alpha[i]
        alpha_cumprod = sched.alpha_cumprod[i]
        x = (1 / torch.sqrt(alpha)) * (x - ((1 - alpha) / torch.sqrt(1 - alpha_cumprod)) * pred_noise)
        if i > 0:
            x = x + torch.sqrt(sched.posterior_variance[i]) * torch.randn_like(x)
    x = (x.clamp(-1, 1) + 1) / 2

    return _to_pil(x[0].permute(1, 2, 0).cpu().numpy())


# _______________________________________________________________
# Diffusion new
def load_diffusion_new():
    if "diff_new" not in _cache:
        sched = LinearNoiseScheduler(timesteps=1000, device=device).to(device)
        m = UNet32New(num_glasses, num_head, num_body).to(device)
        m.load_state_dict(torch.load(WEIGHTS["diffusion_new"], map_location=device))
        m.eval()
        _cache["diff_new"] = (m, sched)

    return _cache["diff_new"]


@torch.no_grad()
def generate_diffusion_new(g, h, b):
    m, sched = load_diffusion_new()
    cond = torch.tensor([[g, h, b]], dtype=torch.long, device=device)
    x = torch.randn(1, m.in_channels, 32, 32, device=device)
    for i in reversed(range(sched.timesteps)):
        t = torch.full((1,), i, device=device, dtype=torch.long)
        pred_noise = m(x, t, cond, drop_cond_mask=None)
        alpha = sched.alpha[i]
        alpha_cumprod = sched.alpha_cumprod[i]
        x = (1 / torch.sqrt(alpha)) * (x - ((1 - alpha) / torch.sqrt(1 - alpha_cumprod)) * pred_noise)
        if i > 0:
            x = x + torch.sqrt(sched.posterior_variance[i]) * torch.randn_like(x)
    x = (x.clamp(-1, 1) + 1) / 2
    return _to_pil(x[0].permute(1, 2, 0).cpu().numpy())


MODEL_REGISTRY = {
    "VAE": generate_vae,
    "GAN (original)": generate_gan_original,
    "GAN (new)": generate_gan_new,
    "Diffusion (original)": generate_diffusion_original,
    "Diffusion (new)": generate_diffusion_new,
}

## Frontend

Własny CSS - żeby nie polegać na zewnetrznych obrazkach. Zmienić??? `image-rendering: pixelated` na galerii, żeby 32x32 nie rozmywało się przy powiekszeniu.

In [ ]:
# MASCOT = "iVBORw0KGgoAAAANSUhEUgAAAMAAAADACAYAAABS3GwHAAACkUlEQVR4nO3dsU0DQRBAUR+iGhKaoBLqIKQOKqEJEtoxNazMam/vvxdb1srS1ySj8XHb3Mfv/b76DWWfL8ex+g2PeFr9AFhJAKQJgDQBkCYA0gRAmgBIEwBpAiBNAKQJgLTT7XHY7bm2s+0OmQCkCYA0AZAmANIEQJoASBMAaQIgTQCkCYA0AZA2fS/Dbg+PmL07ZAKQJgDSBECaAEgTAGkCIE0ApAmANAGQJgDSBEDa8J6F3R7ObHR3yAQgTQCkCYA0AZAmANIEQJoASBMAaQIgTQCkCYA0AZAmANIEQJoASBMAaQIgTQCkCYA0AZAmANIEQJoASHte/YBHjd6BmX3XyHv2YgKQJgDSBECaAEgTAGkCIE0ApAmANAGQJgDSBECaAEgTAGkCIE0ApAmANAGQJgDSBECaAEgTAGkCIE0ApA3djLndendj2MvoHSQTgDQBkCYA0gRAmgBIEwBpAiBNAKQJgDQBkCYA0rb/j7Cvt9ehz79//0x6yTXUfk8TgDQBkCYA0gRAmgBIEwBpAiBNAKQJgDQBkCYA0twF4lLcBYIBAiBNAKQJgDQBkCYA0gRAmgBIEwBpAiBNAKRtfxdottHdklF2q9YyAUgTAGkCIE0ApAmANAGQJgDSBECaAEgTAGkCIM0u0GKju0Z2h/6XCUCaAEgTAGkCIE0ApAmANAGQJgDSBECaAEgTAGnTd4Fm77qcbZdm9vtH1X7/USYAaQIgTQCkCYA0AZAmANIEQJoASBMAaQIgTQCkne4u0O67K7Xvn232e0wA0gRAmgBIEwBpAiBNAKQJgDQBkCYA0gRAmgBIEwBpAiBNAKQJgDQBkCYA0gRAmgBIEwBpAiBNAKQJAAAAAAAAAAAAgO39Ae+qWM9L+V5VAAAAAElFTkSuQmCC"
# BACKGROUND = "iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAAAfElEQVR4nO3YsQ3CABAEQUBkJnIvTlwL/YeOcE4V75WlmQJOWn32z337Pib9zmN0/zW6fgEBNQE1ATUBNQE1ATUBNQE1ATUBNQE1ATUBNQE1AbX39P/+s6yj+7e/gICagJqAmoCagJqAmoCagJqAmoCagJqAmoCagJqA2h/6vgckrTIh5gAAAABJRU5ErkJggg=="
import base64

def load_b64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("ascii")

MASCOT = load_b64(f"{project_path}/data/mascot.png")
BACKGROUND = load_b64(f"{project_path}/data/background.png")


CUSTOM_CSS = f"""
@import url('https://fonts.googleapis.com/css2?family=Press+Start+2P&display=swap');

:root {{
    --bg-deep: #1a1530;
    --bg-panel: #241d3f;
    --accent-cyan: #7cd6ff;
    --accent-blue: #e5dbfecc;
    --accent-yellow: #ffe066;
    --text: #7b7b7b;
}}

.gradio-container {{
    background: var(--bg-deep) !important;
    background-image: url("data:image/png;base64,{BACKGROUND}") !important;
    background-size: cover !important;
    background-position: center !important;
    background-repeat: no-repeat !important;
    background-attachment: fixed !important;
    font-family: 'Press Start 2P', monospace !important;
    color: var(--text) !important;
}}

#pixeloza-header {{
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 20px;
    padding: 18px 10px;
    margin-bottom: 10px;
    background: var(--bg-panel);
    border: 4px solid var(--accent-cyan);
    box-shadow: 6px 6px 0 rgba(0,0,0,0.4);
}}

#pixeloza-header img {{
    image-rendering: pixelated;
    width: 128px;
    height: 128px;
}}

#pixeloza-header h1 {{
    font-family: 'Press Start 2P', monospace;
    font-size: 22px;
    color: var(--accent-yellow);
    text-shadow: 3px 3px 0 #000;
    margin: 0;
}}

#pixeloza-header p {{
    font-family: monospace;
    font-size: 13px;
    color: var(--accent-cyan);
    margin: 4px 0 0 0;
}}

.gr-panel, .gr-box, .gr-form {{
    background: var(--bg-panel) !important;
    border: 3px solid var(--accent-blue) !important;
    box-shadow: 4px 4px 0 rgba(0,0,0,0.35) !important;
}}

/* prostokat z lewej (dropdowny + checkboxy) */
#controls-column {{
    background: rgba(94, 60, 177, 0.8) !important;
    border: 3px solid var(--accent-blue) !important;
    padding: 12px !important;
}}

/* prostokat z prawej (galeria wynikow) */
#output-column {{
    background: rgba(94, 60, 177, 0.8) !important;
    border: 3px solid var(--accent-blue) !important;
}}

/* tlo i tekst kazdego dropdowna */
#glasses-dropdown, #head-dropdown, #body-dropdown {{
    background: rgba(94, 60, 177, 0.8) !important;
}}
#glasses-dropdown input, #head-dropdown input, #body-dropdown input,
#glasses-dropdown span, #head-dropdown span, #body-dropdown span {{
    color: rgba(0, 0, 0, 0.8) !important;
}}

/* caly blok checkboxow (tlo calego "pudelka") */
#model-checkboxes {{
    background: rgba(94, 60, 177, 0.8) !important;
}}
/* kazdy pojedynczy checkbox z osobna - i jego tlo, i tekst */
#model-checkboxes label {{
    background: rgba(94, 60, 177, 0.8) !important;
    color: rgba(0, 0, 0, 0.8) !important;
    border: 2px solid var(--accent-blue) !important;
}}

button.primary {{
    font-family: 'Press Start 2P', monospace !important;
    background: var(--accent-blue) !important;
    color: #1a1530 !important;
    border: 3px solid #000 !important;
    box-shadow: 4px 4px 0 #000 !important;
    font-size: 12px !important;
}}

button.primary:hover {{
    transform: translate(2px, 2px);
    box-shadow: 2px 2px 0 #000 !important;
}}

#output-gallery img {{
    image-rendering: pixelated !important;
}}
"""

In [ ]:
def generate_compare(glasses_name, head_name, body_name, selected_models):
    g = id_to_name["glasses"].index(glasses_name)
    h = id_to_name["head"].index(head_name)
    b = id_to_name["body"].index(body_name)

    results = []
    for name in selected_models:
        try:
            img = MODEL_REGISTRY[name](g, h, b)
            results.append((img, name))
        except Exception as e:
            print(f"Błąd przy {name}: {e}")
            results.append((Image.new("RGBA", (256, 256), (255, 0, 0, 128)), f"{name} - BŁĄD"))
    return results


with gr.Blocks(css=CUSTOM_CSS, title="Pixeloza") as demo:
    gr.HTML(f"""
    <div id="pixeloza-header">
        <img src="data:image/png;base64,{MASCOT}" alt="maskotka"/>
        <div>
            <h1>PIXELOZA</h1>
            <p>Wybierz cechy &gt; Zaznacz modele &gt; Porównaj wyniki</p>
        </div>
    </div>
    """)

    with gr.Row(elem_id="main-row"):
        with gr.Column(scale=1, elem_id="controls-column"):
            glasses_dd = gr.Dropdown(choices=id_to_name["glasses"], label="Okulary", value=id_to_name["glasses"][0], elem_id="glasses-dropdown")
            head_dd = gr.Dropdown(choices=id_to_name["head"], label="Kształt głowy", value=id_to_name["head"][0], elem_id="head-dropdown")
            body_dd = gr.Dropdown(choices=id_to_name["body"], label="Kolor ciała", value=id_to_name["body"][0], elem_id="body-dropdown")
            models_cbg = gr.CheckboxGroup(choices=list(MODEL_REGISTRY.keys()), label="Modele do porównania", value=["VAE"], elem_id="model-checkboxes")
            generate_btn = gr.Button("Generuj", variant="primary")

        with gr.Column(scale=2):
            gallery = gr.Gallery(label="Wyniki", elem_id="output-gallery", columns=5, height="auto", object_fit="contain")

    generate_btn.click(fn=generate_compare, inputs=[glasses_dd, head_dd, body_dd, models_cbg], outputs=gallery)

demo.launch(share=True)